In [1]:
import os

folders = [
    "src",
    "configs",
    "data",
    "reports"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

with open("src/__init__.py", "w", encoding="utf-8") as file:
    pass

print("Project folders created successfully!")

Project folders created successfully!


In [2]:
%%writefile src/utils.py
import logging
import os
import datetime

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)


def setup_logging(name, level=logging.INFO, log_file=None):
    logger = logging.getLogger(name)
    logger.setLevel(level)

    if not logger.handlers:
        formatter = logging.Formatter(
            "%(asctime)s - %(name)s - %(levelname)s - %(message)s"
        )

        console_handler = logging.StreamHandler()
        console_handler.setFormatter(formatter)
        logger.addHandler(console_handler)

        if log_file:
            os.makedirs(os.path.dirname(log_file) or ".", exist_ok=True)
            file_handler = logging.FileHandler(
                log_file, encoding="utf-8"
            )
            file_handler.setFormatter(formatter)
            logger.addHandler(file_handler)

    return logger


def ensure_directory_exists(path):
    if not path:
        return

    try:
        os.makedirs(path, exist_ok=True)
    except OSError as error:
        logging.error("Error creating directory %s: %s", path, error)
        raise


def log_audit_entry(
    audit_log_list,
    status,
    message,
    task_type="N/A",
    details=None
):
    entry = {
        "timestamp": datetime.datetime.now().isoformat(),
        "status": status,
        "task_type": task_type,
        "message": message,
        "details": details if details is not None else {}
    }

    audit_log_list.append(entry)

    if status in ("ERROR", "CRITICAL"):
        logging.error("AUDIT: %s - %s", status, message)
    else:
        logging.info("AUDIT: %s - %s", status, message)

Writing src/utils.py


In [3]:
%%writefile src/config_parser.py
import json
import os
from .utils import setup_logging

logger = setup_logging(__name__)


class ConfigParser:
    def __init__(self, config_path):
        self.config_path = config_path
        self.config = {}

    def load_config(self):
        if not os.path.exists(self.config_path):
            raise FileNotFoundError(
                f"Configuration file not found: {self.config_path}"
            )

        try:
            with open(
                self.config_path, "r", encoding="utf-8"
            ) as file:
                self.config = json.load(file)

        except json.JSONDecodeError as error:
            raise ValueError(
                f"Invalid JSON configuration: {error}"
            ) from error

        self._validate_config()
        logger.info("Configuration loaded and validated.")
        return self.config

    def _validate_config(self):
        if not isinstance(self.config, dict):
            raise ValueError("Configuration must be a JSON object.")

        if not isinstance(self.config.get("tasks"), list):
            raise ValueError("'tasks' must be a list.")

        for index, task in enumerate(self.config["tasks"]):
            if not isinstance(task, dict):
                raise ValueError(f"Task {index} must be an object.")

            if not task.get("type"):
                raise ValueError(f"Task {index} is missing 'type'.")

            if not task.get("output_data_key"):
                raise ValueError(
                    f"Task {index} is missing 'output_data_key'."
                )

            task_type = task["type"]

            if task_type in ("load_csv", "load_json", "save_csv", "save_json"):
                if not task.get("file_path"):
                    raise ValueError(
                        f"Task {index} requires 'file_path'."
                    )

            if task_type in (
                "save_csv",
                "save_json",
                "transform_csv_aggregate",
                "filter_csv",
                "update_json_data",
                "query_json_data",
                "csv_to_json_conversion",
                "json_to_csv_conversion"
            ):
                if not task.get("input_data_key"):
                    raise ValueError(
                        f"Task {index} requires 'input_data_key'."
                    )

        logger.info("Configuration validation completed.")

Writing src/config_parser.py


In [4]:
%%writefile src/csv_tasks.py
import csv
import os
from collections import defaultdict
from .utils import setup_logging, ensure_directory_exists

logger = setup_logging(__name__)


class CSVTasks:
    def load_csv(self, file_path):
        if not os.path.exists(file_path):
            logger.error("CSV file not found: %s", file_path)
            return None

        try:
            with open(
                file_path, "r", newline="", encoding="utf-8"
            ) as file:
                data = list(csv.DictReader(file))

            logger.info("Loaded %s CSV rows.", len(data))
            return data

        except (OSError, csv.Error) as error:
            logger.error("Cannot read CSV: %s", error)
            return None

    def save_csv(self, data, file_path, fieldnames=None):
        if not isinstance(data, list):
            logger.error("CSV data must be a list.")
            return False

        if not data:
            logger.warning("No data to save to CSV.")
            return False

        if fieldnames is None:
            if not isinstance(data[0], dict):
                return False
            fieldnames = list(data[0].keys())

        ensure_directory_exists(os.path.dirname(file_path))

        try:
            with open(
                file_path, "w", newline="", encoding="utf-8"
            ) as file:
                writer = csv.DictWriter(
                    file,
                    fieldnames=fieldnames,
                    extrasaction="ignore"
                )
                writer.writeheader()
                writer.writerows(data)

            logger.info("Saved %s rows to %s.", len(data), file_path)
            return True

        except (OSError, csv.Error) as error:
            logger.error("Cannot save CSV: %s", error)
            return False

    def transform_csv_aggregate(
        self, data, group_by_field, aggregate_field,
        aggregate_type="sum"
    ):
        if not data:
            return []

        groups = defaultdict(lambda: {"sum": 0.0, "count": 0})

        for row in data:
            group_value = row.get(group_by_field)

            if group_value is None or group_value == "":
                logger.warning("Skipping row without grouping field: %s", row)
                continue

            try:
                number = float(row[aggregate_field])
            except (ValueError, TypeError, KeyError):
                logger.warning("Skipping invalid numeric row: %s", row)
                continue

            groups[group_value]["sum"] += number
            groups[group_value]["count"] += 1

        results = []

        for group_value, metrics in groups.items():
            result = {group_by_field: group_value}

            if aggregate_type == "sum":
                result[f"Total_{aggregate_field}"] = round(
                    metrics["sum"], 2
                )
            elif aggregate_type == "count":
                result[f"Count_{aggregate_field}"] = metrics["count"]
            elif aggregate_type == "average":
                result[f"Average_{aggregate_field}"] = round(
                    metrics["sum"] / metrics["count"], 2
                )
            else:
                raise ValueError(
                    f"Unsupported aggregate type: {aggregate_type}"
                )

            results.append(result)

        logger.info("Aggregation produced %s groups.", len(results))
        return results

    def filter_csv(self, data, filter_field, operator, value):
        operators = {
            ">": lambda a, b: a > b,
            "<": lambda a, b: a < b,
            ">=": lambda a, b: a >= b,
            "<=": lambda a, b: a <= b,
            "==": lambda a, b: a == b,
            "!=": lambda a, b: a != b
        }

        if operator not in operators:
            raise ValueError(f"Unsupported operator: {operator}")

        filtered = []

        for row in data or []:
            field_value = row.get(filter_field)

            if field_value is None:
                continue

            try:
                if isinstance(value, (int, float)) and not isinstance(value, bool):
                    left = float(field_value)
                    right = float(value)
                else:
                    left = str(field_value)
                    right = str(value)

                if operators[operator](left, right):
                    filtered.append(row)

            except (ValueError, TypeError):
                logger.warning("Cannot compare row: %s", row)

        logger.info("Filter returned %s rows.", len(filtered))
        return filtered

Writing src/csv_tasks.py


In [5]:
%%writefile src/json_tasks.py
import json
import os
from .utils import setup_logging, ensure_directory_exists

logger = setup_logging(__name__)


class JSONTasks:
    def load_json(self, file_path):
        if not os.path.exists(file_path):
            logger.error("JSON file not found: %s", file_path)
            return None

        try:
            with open(file_path, "r", encoding="utf-8") as file:
                return json.load(file)

        except (OSError, json.JSONDecodeError) as error:
            logger.error("Cannot read JSON: %s", error)
            return None

    def save_json(self, data, file_path, indent=2):
        ensure_directory_exists(os.path.dirname(file_path))

        try:
            with open(file_path, "w", encoding="utf-8") as file:
                json.dump(
                    data, file, indent=indent, ensure_ascii=False
                )

            logger.info("Saved JSON to %s.", file_path)
            return True

        except (OSError, TypeError, ValueError) as error:
            logger.error("Cannot save JSON: %s", error)
            return False

    def update_json_data(self, data, updates):
        if not isinstance(updates, list):
            logger.error("Updates must be a list.")
            return None

        # Deep copy เพื่อไม่แก้ข้อมูลต้นฉบับโดยตรง
        modified = json.loads(json.dumps(data))

        for update in updates:
            path = update.get("path", "")
            value = update.get("value")
            operation = update.get("operation", "set")

            # กรณีเพิ่มสินค้าใหม่ลงในรายการ JSON ชั้นบนสุด
            if path == "" and operation == "add_to_list":
                if isinstance(modified, list):
                    modified.append(value)
                    logger.info("Added an item to the inventory list.")
                else:
                    logger.warning("Root data is not a list.")
                continue

            if not path:
                logger.warning("Skipping update with an empty path.")
                continue

            keys = path.split(".")
            current = modified
            valid = True

            # เดินทางไปยังโครงสร้างแม่ของข้อมูลเป้าหมาย
            for key in keys[:-1]:
                if isinstance(current, dict) and key in current:
                    current = current[key]
                elif isinstance(current, list) and key.isdigit():
                    index = int(key)
                    if 0 <= index < len(current):
                        current = current[index]
                    else:
                        valid = False
                        break
                else:
                    valid = False
                    break

            if not valid:
                logger.warning("Update path not found: %s", path)
                continue

            target = keys[-1]

            if isinstance(current, dict):
                if operation == "set":
                    current[target] = value

                elif operation == "add_to_list":
                    if isinstance(current.get(target), list):
                        current[target].append(value)
                    else:
                        logger.warning("Target is not a list: %s", path)

                elif operation == "remove_from_list":
                    if isinstance(current.get(target), list):
                        if value in current[target]:
                            current[target].remove(value)
                        else:
                            logger.warning("Value not found at %s", path)
                    else:
                        logger.warning("Target is not a list: %s", path)

                else:
                    logger.warning("Unsupported operation: %s", operation)

            elif isinstance(current, list) and target.isdigit():
                index = int(target)

                if 0 <= index < len(current) and operation == "set":
                    current[index] = value
                else:
                    logger.warning("Invalid list update: %s", path)

            else:
                logger.warning("Cannot update path: %s", path)

        return modified

    def query_json_data(self, data, path):
        if not path:
            return data

        current = data

        try:
            for key in path.split("."):
                if isinstance(current, dict):
                    current = current.get(key)
                elif isinstance(current, list) and key.isdigit():
                    index = int(key)
                    if index >= len(current):
                        return None
                    current = current[index]
                else:
                    return None

                if current is None:
                    return None

            logger.info("Query '%s' returned: %s", path, current)
            return current

        except (IndexError, TypeError, ValueError):
            return None

Writing src/json_tasks.py


In [6]:
%%writefile src/conversion_tasks.py
import csv
import json
from .utils import setup_logging

logger = setup_logging(__name__)


class ConversionTasks:
    def csv_to_json(self, csv_data):
        if not csv_data:
            return []

        converted = []

        for row in csv_data:
            new_row = {}

            for key, value in row.items():
                text = "" if value is None else str(value).strip()

                if text == "" or text.lower() == "null":
                    new_row[key] = None
                elif text.lower() in ("true", "false"):
                    new_row[key] = text.lower() == "true"
                else:
                    try:
                        number = float(text)
                        new_row[key] = (
                            int(number) if number.is_integer() else number
                        )
                    except ValueError:
                        new_row[key] = text

            converted.append(new_row)

        logger.info("Converted CSV rows to JSON-compatible data.")
        return converted

    def json_to_csv(self, json_data, fieldnames=None):
        if not isinstance(json_data, list) or not json_data:
            return [], fieldnames or []

        # รวบรวมชื่อคอลัมน์ รวมถึง key ใน dict ที่ซ้อนอยู่
        if fieldnames is None:
            keys = []

            def collect_keys(item, prefix=""):
                if isinstance(item, dict):
                    for key, value in item.items():
                        full_key = f"{prefix}.{key}" if prefix else key

                        if isinstance(value, dict):
                            collect_keys(value, full_key)
                        else:
                            if full_key not in keys:
                                keys.append(full_key)

            for item in json_data:
                collect_keys(item)

            fieldnames = keys

        rows = []

        for item in json_data:
            row = {}

            for field in fieldnames:
                value = item
                found = True

                for part in field.split("."):
                    if isinstance(value, dict) and part in value:
                        value = value[part]
                    else:
                        found = False
                        break

                if not found:
                    value = ""

                if isinstance(value, (dict, list)):
                    value = json.dumps(value, ensure_ascii=False)

                row[field] = value

            rows.append(row)

        logger.info("Converted JSON data to CSV-compatible rows.")
        return rows, fieldnames

Writing src/conversion_tasks.py


In [7]:
%%writefile src/data_agent.py
import os
import json
import datetime

from .utils import (
    setup_logging,
    log_audit_entry,
    ensure_directory_exists
)
from .csv_tasks import CSVTasks
from .json_tasks import JSONTasks
from .conversion_tasks import ConversionTasks

logger = setup_logging(__name__)


class DataAgent:
    def __init__(self, config):
        self.config = config
        self.audit_log = []
        self.data_store = {}

        self.csv_tasks = CSVTasks()
        self.json_tasks = JSONTasks()
        self.conversion_tasks = ConversionTasks()

        logger.info("DataAgent initialized.")

    def _process_task(self, task):
        task_type = task["type"]
        task_name = task.get("name", task_type)
        input_key = task.get("input_data_key")
        output_key = task["output_data_key"]

        log_audit_entry(
            self.audit_log,
            "INFO",
            f"Starting task: {task_name}",
            task_type,
            task
        )

        try:
            current_data = None

            if input_key:
                if input_key not in self.data_store:
                    raise KeyError(
                        f"Input data key not found: {input_key}"
                    )
                current_data = self.data_store[input_key]

            output = None
            success = False

            if task_type == "load_csv":
                output = self.csv_tasks.load_csv(task["file_path"])
                success = output is not None

            elif task_type == "save_csv":
                success = self.csv_tasks.save_csv(
                    current_data,
                    task["file_path"],
                    task.get("fieldnames")
                )
                output = current_data

            elif task_type == "transform_csv_aggregate":
                output = self.csv_tasks.transform_csv_aggregate(
                    current_data,
                    task["group_by_field"],
                    task["aggregate_field"],
                    task.get("aggregate_type", "sum")
                )
                success = output is not None

            elif task_type == "filter_csv":
                output = self.csv_tasks.filter_csv(
                    current_data,
                    task["filter_field"],
                    task["operator"],
                    task["value"]
                )
                success = output is not None

            elif task_type == "load_json":
                output = self.json_tasks.load_json(task["file_path"])
                success = output is not None

            elif task_type == "save_json":
                success = self.json_tasks.save_json(
                    current_data,
                    task["file_path"],
                    task.get("indent", 2)
                )
                output = current_data

            elif task_type == "update_json_data":
                output = self.json_tasks.update_json_data(
                    current_data,
                    task.get("updates", [])
                )
                success = output is not None

            elif task_type == "query_json_data":
                output = self.json_tasks.query_json_data(
                    current_data,
                    task["path"]
                )
                success = output is not None

            elif task_type == "csv_to_json_conversion":
                output = self.conversion_tasks.csv_to_json(current_data)
                success = output is not None

            elif task_type == "json_to_csv_conversion":
                output, _ = self.conversion_tasks.json_to_csv(
                    current_data,
                    task.get("fieldnames")
                )
                success = output is not None

            else:
                raise ValueError(f"Unsupported task type: {task_type}")

            if not success:
                raise RuntimeError(f"Task failed: {task_name}")

            if output_key:
                self.data_store[output_key] = output

            log_audit_entry(
                self.audit_log,
                "SUCCESS",
                f"Completed task: {task_name}",
                task_type,
                {"output_data_key": output_key}
            )

        except Exception as error:
            logger.exception("Task failed: %s", task_name)

            log_audit_entry(
                self.audit_log,
                "ERROR",
                f"Task failed: {task_name}",
                task_type,
                {"error": str(error)}
            )

    def _generate_audit_report(self, output_dir="reports"):
        ensure_directory_exists(output_dir)

        timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        report_path = os.path.join(
            output_dir, f"audit_report_{timestamp}.json"
        )

        with open(report_path, "w", encoding="utf-8") as file:
            json.dump(
                self.audit_log,
                file,
                indent=2,
                ensure_ascii=False
            )

        logger.info("Audit report saved to %s.", report_path)
        return report_path

    def run(self):
        start_time = datetime.datetime.now()

        log_audit_entry(
            self.audit_log,
            "INFO",
            "Data Agent started.",
            "agent_run"
        )

        for task in self.config.get("tasks", []):
            self._process_task(task)

        duration = (
            datetime.datetime.now() - start_time
        ).total_seconds()

        log_audit_entry(
            self.audit_log,
            "INFO",
            f"Data Agent finished in {duration:.2f} seconds.",
            "agent_run_summary"
        )

        report_path = self._generate_audit_report()

        return {
            "data_store": self.data_store,
            "audit_log": self.audit_log,
            "report_path": report_path
        }

Writing src/data_agent.py


In [8]:
%%writefile configs/data_pipeline_config.json
{
    "tasks": [
        {
            "name": "Load Sales Data CSV",
            "type": "load_csv",
            "file_path": "data/input_sales.csv",
            "output_data_key": "raw_sales_data"
        },
        {
            "name": "Aggregate Sales by Customer",
            "type": "transform_csv_aggregate",
            "input_data_key": "raw_sales_data",
            "output_data_key": "customer_sales_summary",
            "group_by_field": "Customer",
            "aggregate_field": "Amount",
            "aggregate_type": "sum"
        },
        {
            "name": "Save Aggregated Sales CSV",
            "type": "save_csv",
            "file_path": "data/processed_sales_by_customer.csv",
            "input_data_key": "customer_sales_summary",
            "output_data_key": "saved_sales_summary",
            "fieldnames": ["Customer", "Total_Amount"]
        },
        {
            "name": "Convert Raw Sales to JSON",
            "type": "csv_to_json_conversion",
            "input_data_key": "raw_sales_data",
            "output_data_key": "raw_sales_json"
        },
        {
            "name": "Save Raw Sales JSON",
            "type": "save_json",
            "file_path": "data/sales_data.json",
            "input_data_key": "raw_sales_json",
            "output_data_key": "saved_sales_json",
            "indent": 4
        },
        {
            "name": "Load Inventory Data JSON",
            "type": "load_json",
            "file_path": "data/input_inventory.json",
            "output_data_key": "raw_inventory_data"
        },
        {
            "name": "Update Inventory Stock and Add Product",
            "type": "update_json_data",
            "input_data_key": "raw_inventory_data",
            "output_data_key": "updated_inventory_data",
            "updates": [
                {
                    "path": "0.stock",
                    "value": 75,
                    "operation": "set"
                },
                {
                    "path": "1.details.layout",
                    "value": "UK ISO",
                    "operation": "set"
                },
                {
                    "path": "2.details.dpi",
                    "value": 2000,
                    "operation": "set"
                },
                {
                    "path": "",
                    "value": {
                        "id": "PROD004",
                        "name": "Smart Speaker",
                        "category": "Audio",
                        "stock": 40,
                        "details": {
                            "brand": "SoundGenius",
                            "features": [
                                "Voice Assistant",
                                "Multi-room"
                            ]
                        }
                    },
                    "operation": "add_to_list"
                },
                {
                    "path": "0.details.color",
                    "value": "Space Grey",
                    "operation": "set"
                }
            ]
        },
        {
            "name": "Query Specific Product Brand",
            "type": "query_json_data",
            "input_data_key": "updated_inventory_data",
            "output_data_key": "queried_product_brand",
            "path": "0.details.brand"
        },
        {
            "name": "Save Updated Inventory JSON",
            "type": "save_json",
            "file_path": "data/updated_inventory.json",
            "input_data_key": "updated_inventory_data",
            "output_data_key": "saved_inventory",
            "indent": 2
        }
    ]
}

Writing configs/data_pipeline_config.json


In [9]:
%%writefile data/input_sales.csv
OrderID,Product,Amount,Price,Customer
101,Laptop,1200.50,1200.50,Alice
102,Mouse,25.00,25.00,Bob
103,Keyboard,75.25,75.25,Alice
104,Monitor,300.00,300.00,Charlie
105,Webcam,45.75,45.75,Bob
106,External HDD,90.00,90.00,David
107,Headphones,150.00,150.00,Alice
108,USB Hub,15.50,15.50,Eve
109,Gaming Mouse,60.00,60.00,Bob
110,SSD,200.00,200.00,Charlie

Writing data/input_sales.csv


In [10]:
%%writefile data/input_inventory.json
[
    {
        "id": "PROD001",
        "name": "Laptop Pro",
        "category": "Electronics",
        "stock": 50,
        "details": {
            "brand": "TechCorp",
            "model": "XPS 15"
        }
    },
    {
        "id": "PROD002",
        "name": "Mechanical Keyboard",
        "category": "Peripherals",
        "stock": 120,
        "details": {
            "brand": "KeyMaster",
            "layout": "US ANSI"
        }
    },
    {
        "id": "PROD003",
        "name": "Ergonomic Mouse",
        "category": "Peripherals",
        "stock": 80,
        "details": {
            "brand": "ErgoGear",
            "dpi": 1600
        }
    }
]

Writing data/input_inventory.json


In [11]:
%%writefile main.py
import os

from src.config_parser import ConfigParser
from src.data_agent import DataAgent
from src.utils import setup_logging, ensure_directory_exists

logger = setup_logging(__name__)


def main():
    base_dir = os.path.abspath(".")
    config_path = os.path.join(
        base_dir, "configs", "data_pipeline_config.json"
    )

    ensure_directory_exists(os.path.join(base_dir, "data"))
    ensure_directory_exists(os.path.join(base_dir, "configs"))
    ensure_directory_exists(os.path.join(base_dir, "reports"))

    parser = ConfigParser(config_path)
    config = parser.load_config()

    # ทำให้ทุก file_path อ้างอิงจากโฟลเดอร์โปรเจกต์
    for task in config["tasks"]:
        if "file_path" in task:
            task["file_path"] = os.path.join(
                base_dir, task["file_path"]
            )

    # ตรวจสอบไฟล์ต้นทางก่อนเริ่มประมวลผล
    for task in config["tasks"]:
        if task["type"] in ("load_csv", "load_json"):
            if not os.path.exists(task["file_path"]):
                raise FileNotFoundError(
                    f"Input file not found: {task['file_path']}"
                )

    logger.info("Initializing Data Agent...")
    agent = DataAgent(config)

    logger.info("Running data pipeline...")
    result = agent.run()

    print("\n========== PIPELINE SUMMARY ==========")
    print("Tasks recorded:", len(result["audit_log"]))
    print("Data store keys:", list(result["data_store"].keys()))
    print("Audit report:", result["report_path"])

    print(
        "Queried product brand:",
        result["data_store"].get("queried_product_brand")
    )

    print("\nPipeline completed.")


if __name__ == "__main__":
    main()

Writing main.py


In [12]:
!python main.py

2026-10-10 22:24:05,754 - src.config_parser - INFO - Configuration validation completed.
2026-10-10 22:24:05,754 - INFO - Configuration validation completed.
2026-10-10 22:24:05,755 - src.config_parser - INFO - Configuration loaded and validated.
2026-10-10 22:24:05,755 - INFO - Configuration loaded and validated.
2026-10-10 22:24:05,755 - __main__ - INFO - Initializing Data Agent...
2026-10-10 22:24:05,755 - INFO - Initializing Data Agent...
2026-10-10 22:24:05,755 - src.data_agent - INFO - DataAgent initialized.
2026-10-10 22:24:05,755 - INFO - DataAgent initialized.
2026-10-10 22:24:05,755 - __main__ - INFO - Running data pipeline...
2026-10-10 22:24:05,755 - INFO - Running data pipeline...
2026-10-10 22:24:05,755 - INFO - AUDIT: INFO - Data Agent started.
2026-10-10 22:24:05,755 - INFO - AUDIT: INFO - Starting task: Load Sales Data CSV
2026-10-10 22:24:05,755 - src.csv_tasks - INFO - Loaded 10 CSV rows.
2026-10-10 22:24:05,755 - INFO - Loaded 10 CSV rows.
2026-10-10 22:24:05,755 - 

In [13]:
import pandas as pd
from IPython.display import display

result_csv = "data/processed_sales_by_customer.csv"

display(pd.read_csv(result_csv))

,Customer,Total_Amount
0,Alice,1425.75
1,Bob,130.75
2,Charlie,500.00
3,David,90.00
4,Eve,15.50


In [14]:
import json
import pandas as pd
from IPython.display import display

with open(
    "data/updated_inventory.json", "r", encoding="utf-8"
) as file:
    updated_inventory = json.load(file)

display(pd.json_normalize(updated_inventory))

,id,name,category,stock,details.brand,details.model,details.color,details.layout,details.dpi,details.features
0,PROD001,Laptop Pro,Electronics,75,TechCorp,XPS 15,Space Grey,NaN,NaN,NaN
1,PROD002,Mechanical Keyboard,Peripherals,120,KeyMaster,NaN,NaN,UK ISO,NaN,NaN
2,PROD003,Ergonomic Mouse,Peripherals,80,ErgoGear,NaN,NaN,NaN,2000.0,NaN
3,PROD004,Smart Speaker,Audio,40,SoundGenius,NaN,NaN,NaN,NaN,"[Voice Assistant, Multi-room]"


In [15]:
import glob
import json
import pandas as pd
from IPython.display import display

report_files = sorted(glob.glob("reports/audit_report_*.json"))

if report_files:
    latest_report = report_files[-1]

    with open(latest_report, "r", encoding="utf-8") as file:
        audit_data = json.load(file)

    print("Audit report:", latest_report)
    display(pd.DataFrame(audit_data))
else:
    print("ไม่พบ Audit Report กรุณารัน Cell 12 ก่อน")

Audit report: reports/audit_report_20261010_222405.json


,timestamp,status,task_type,message,details
0,2026-10-10T22:24:05.755471,INFO,agent_run,Data Agent started.,{}
1,2026-10-10T22:24:05.755568,INFO,load_csv,Starting task: Load Sales Data CSV,"{'name': 'Load Sales Data CSV', 'type': 'load_..."
2,2026-10-10T22:24:05.755883,SUCCESS,load_csv,Completed task: Load Sales Data CSV,{'output_data_key': 'raw_sales_data'}
3,2026-10-10T22:24:05.755944,INFO,transform_csv_aggregate,Starting task: Aggregate Sales by Customer,"{'name': 'Aggregate Sales by Customer', 'type'..."
4,2026-10-10T22:24:05.756157,SUCCESS,transform_csv_aggregate,Completed task: Aggregate Sales by Customer,{'output_data_key': 'customer_sales_summary'}
5,2026-10-10T22:24:05.756211,INFO,save_csv,Starting task: Save Aggregated Sales CSV,"{'name': 'Save Aggregated Sales CSV', 'type': ..."
6,2026-10-10T22:24:05.756538,SUCCESS,save_csv,Completed task: Save Aggregated Sales CSV,{'output_data_key': 'saved_sales_summary'}
7,2026-10-10T22:24:05.756592,INFO,csv_to_json_conversion,Starting task: Convert Raw Sales to JSON,"{'name': 'Convert Raw Sales to JSON', 'type': ..."
8,2026-10-10T22:24:05.756805,SUCCESS,csv_to_json_conversion,Completed task: Convert Raw Sales to JSON,{'output_data_key': 'raw_sales_json'}
9,2026-10-10T22:24:05.756857,INFO,save_json,Starting task: Save Raw Sales JSON,"{'name': 'Save Raw Sales JSON', 'type': 'save_..."


In [16]:
from google.colab import files

files.download("data/processed_sales_by_customer.csv")
files.download("data/sales_data.json")
files.download("data/updated_inventory.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>